In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
sys.path.append('../../')  # Add project root to path

import logging
from modules.base_ingestion import BaseBronzeIngestion
from typing import List, Dict, Any

# Log Configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class FrenteMembrosIngestion(BaseBronzeIngestion):
    """Concrete implementation for Frente/Membros ingestion."""

    def __init__(self, spark, entity_name: str = 'frentes_membros'):
        """Initialize calling base class with only Spark and entity name."""
        # Pass spark and entity name to initialize base class
        super().__init__(
            spark=spark,
            entity_name=entity_name
        )
    
    def fetch_data(self) -> List[Dict[str, Any]]:
        """Implements the abstract method - uses _fetch_multithreaded from base class."""
        logger.info(f"Starting {self.config.entity} ingestion pipeline...")
        
        # Use multithreaded extraction to fetch members for each frente
        return self._fetch_multithreaded(
            source_table=f"{self.generic_config['catalog']}.camara_{self.generic_config['layer']}.frentes",
            id_column='id',
            endpoint_builder=lambda frente_id: f"frentes/{frente_id}/membros",
            foreign_key='id_frente',
            params={},  # No params for individual frente/membros queries
            paginated=False  # Individual endpoints don't support pagination
        )

# Create ingestion instance
ingestion = FrenteMembrosIngestion(spark=spark)

In [0]:
try:
    ingestion.execute()
    logger.info(f"{ingestion.config.entity} ingestion completed successfully!")
except Exception as e:
    logger.error(f"{ingestion.config.entity} ingestion failed: {str(e)}")
    raise

In [0]:
%sql 
SELECT count(*) FROM workspace.camara_bronze.frentes_membros

In [0]:
%sql 
SELECT * 
FROM workspace.camara_bronze.frentes_membros